# Synthetic lending portfolio analysis in R
By Wael El Ghazzawi. Developed with OpenAI Codex assistance.

This base-R companion explores the published synthetic loan panel, checks its integrity, and reports **account-count** delinquency rates over time. No additional packages are required.

All records and USD amounts are fictional. Scenario labels are hand-set simulation settings, not borrower risk grades. This is an educational data-quality and visualization exercise, not a credit-risk model or forecast. Scheduled principal is not an accounting balance.

Dataset: https://www.kaggle.com/datasets/waelelghazzawi/synthetic-lending-portfolio-and-monthly-payments

Generator: https://github.com/identity-wael/synthetic-lending-playbook

Notebook license: Apache-2.0. Dataset license: CC0-1.0.


In [ ]:
cat("R version:", R.version.string, "\n")
root <- if (dir.exists("/kaggle/input")) "/kaggle/input" else "synthetic_lending"
files <- list.files(root, pattern = "^loans[.]csv$", recursive = TRUE, full.names = TRUE)
stopifnot(length(files) == 1L)
folder <- dirname(files[[1]])
loans <- read.csv(file.path(folder, "loans.csv"), stringsAsFactors = FALSE)
monthly <- read.csv(file.path(folder, "monthly_performance.csv"), stringsAsFactors = FALSE, na.strings = "")
loans$origination_date <- as.Date(loans$origination_date)
monthly$observation_date <- as.Date(monthly$observation_date)
monthly$oldest_unpaid_due_date <- as.Date(monthly$oldest_unpaid_due_date)
stopifnot(nrow(loans) == 1500L, nrow(monthly) == 22743L,
          !anyDuplicated(loans$loan_id),
          !anyDuplicated(monthly[c("loan_id", "observation_date")]),
          setequal(monthly$loan_id, loans$loan_id),
          !anyNA(monthly$observation_date), all(monthly$dpd >= 0))
expected_dpd <- as.integer(monthly$observation_date - monthly$oldest_unpaid_due_date)
expected_dpd[is.na(expected_dpd)] <- 0L
stopifnot(all(expected_dpd == monthly$dpd),
          all(is.na(monthly$oldest_unpaid_due_date) == (monthly$unpaid_installments == 0)))
cat("PASS: keys, row counts, relationships and calendar DPD\n")


## Latest snapshot: explicit denominators
Use each account's observation at the common latest cutoff. Every loan must be present exactly once. A rate is the number of accounts at least 30 days overdue divided by all accounts in that scenario. It is not a dollar-weighted loss rate.


In [ ]:
cutoff <- max(monthly$observation_date)
snapshot <- merge(monthly[monthly$observation_date == cutoff, ], loans, by = "loan_id")
stopifnot(nrow(snapshot) == nrow(loans), !anyDuplicated(snapshot$loan_id))
scenarios <- sort(unique(snapshot$segment))
latest <- do.call(rbind, lapply(scenarios, function(s) {
  x <- snapshot[snapshot$segment == s, ]
  data.frame(scenario = s, accounts = nrow(x), overdue_30_plus = sum(x$dpd >= 30),
             overdue_share = mean(x$dpd >= 30))
}))
stopifnot(sum(latest$accounts) == nrow(loans), all(latest$overdue_share >= 0 & latest$overdue_share <= 1))
print(latest)
barplot(100 * latest$overdue_share, names.arg = latest$scenario, col = "#1976A3",
        ylab = "Accounts 30+ days past due (%)", main = paste("Synthetic scenarios at", cutoff))


## Monthly trend and payment activity
The denominator changes as loans enter the panel. Consequently this is a changing-portfolio description, not a fixed-cohort causal comparison. Payment receipts are simulated whole installments and can include catching up on older unpaid installments.


In [ ]:
dates <- sort(unique(monthly$observation_date))
trend <- do.call(rbind, lapply(seq_along(dates), function(i) {
  d <- dates[i]
  x <- monthly[monthly$observation_date == d, ]
  data.frame(observation_date = d, accounts = nrow(x),
             overdue_30_plus = sum(x$dpd >= 30), overdue_share = mean(x$dpd >= 30),
             simulated_payments_usd = sum(x$payment_received_usd))
}))
stopifnot(sum(trend$accounts) == nrow(monthly),
          all(trend$overdue_30_plus <= trend$accounts),
          isTRUE(all.equal(tail(trend$overdue_share, 1), mean(snapshot$dpd >= 30))))
old_par <- par(mfrow = c(2, 1), mar = c(4, 4, 3, 1))
plot(trend$observation_date, 100 * trend$overdue_share, type = "o", pch = 16, col = "#1976A3",
     xlab = "Observation month", ylab = "Accounts overdue (%)", main = "30+ DPD: changing portfolio")
plot(trend$observation_date, trend$simulated_payments_usd / 1e6, type = "o", pch = 16, col = "#27836A",
     xlab = "Observation month", ylab = "Fictional USD millions", main = "Simulated payment receipts")
par(old_par)
print(tail(trend, 6))
write.csv(latest, "scenario_snapshot_r.csv", row.names = FALSE)
write.csv(trend, "monthly_portfolio_trend_r.csv", row.names = FALSE)
cat("PASS: snapshot denominators, trend reconciliation and CSV exports\n")


## Interpretation and extensions
The difference between scenarios is built into the generator; it is not evidence that a real customer group is riskier. There are no charge-offs, closures, partial payments or real borrower characteristics. Do not use these outputs to support real lending decisions.

For a useful extension, compare origination cohorts at the same months-on-book and display both numerator and denominator. Explain how a fixed-cohort rate differs from the changing-portfolio chart above.

References: [base R documentation](https://stat.ethz.ch/R-manual/R-release/library/base/html/00Index.html), [R graphics](https://stat.ethz.ch/R-manual/R-release/library/graphics/html/00Index.html).
